In [0]:
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("merge_key", "", "Merge Key")
dbutils.widgets.text("lode_type", "", "Lode Type")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_key = dbutils.widgets.get("merge_key")
lode_type = dbutils.widgets.get("lode_type")

In [0]:
bronze_df = spark.read.table(f"careplus_db_ws.bronze.{table_name}").filter(f"landing_timestamp = '{landing_timestamp}'")


In [0]:
from pyspark.sql import functions as f
updated_df = bronze_df.withColumn("insert_timestamp", f.current_timestamp()).withColumn("ingest_timestamp", f.current_timestamp())

In [0]:
from delta.tables import DeltaTable

silver_table = f"careplus_db_ws.silver.{table_name}"

if not lode_type or lode_type.upper() == "FULL":
    updated_df.write.mode("overwrite").saveAsTable(silver_table)

elif lode_type.upper() == "APPEND":
    updated_df.write.mode("append").saveAsTable(silver_table)

elif lode_type.upper() == "MERGE":
    if spark.catalog.tableExists(silver_table):
        delta_table = DeltaTable.forName(spark, silver_table)
        merge_keys_list = [k.strip() for k in merge_key.split(",")]
        merge_condition = " AND ".join(
            [f"target.{k} = source.{k}" for k in merge_keys_list]
        )
        # Preserve original insert_timestamp for existing records
        update_cols = {c: f"source.{c}" for c in updated_df.columns if c != "insert_timestamp"}

        (delta_table.alias("target")
            .merge(updated_df.alias("source"), merge_condition)
            .whenMatchedUpdate(set=update_cols)
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        # Table doesn't exist yet — create it with initial load
        updated_df.write.saveAsTable(silver_table)

else:
    raise ValueError(f"Unsupported lode_type: '{lode_type}'. Expected 'FULL', 'APPEND', or 'MERGE'.")

In [0]:
print(type(updated_df))
print(table_name)
print(f"careplus_db_ws.silver.{table_name}")

In [0]:
dbutils.notebook.exit(f"Record count for {table_name}: {updated_df.count()}")